# ARCS v0.6.3 — 10 Hz to 10 THz, with 100 solver frequencies

Every principal geometry sweep below uses **100 logarithmically spaced frequencies from 10 Hz to 10¹³ Hz**. Longitudinal impedance and complete dipolar/quadrupolar tensors are calculated directly by full Maxwell BEM. Raw signed real and imaginary components are exported to CSV. Progress messages identify each completed sweep.

Circular cylindrical Maxwell modes check the numerical solution for the same constitutive model; they are not a claim of experimental accuracy across this whole band. The default example uses constant conductivity, μr = εr = 1, and zero relaxation time. Set physically appropriate material properties for an application. The supplied reduced roughness ODE is not used as a full-band reference.

The mesh is a set of boundary line panels. The GUI now keeps the numbers of boundary meshes, panels, and nodes visible above the canvas, including when the overlay is hidden.

ARCS v0.6.0 includes indirect space charge and changes the complex convention. The previous complex-Bessel underflow fix is retained. Restart the kernel after installing, then Run All. All Green-function calls use the exact scaled-Bessel identity; analytical impedances never enter the BEM solve.

**v0.6.3 output provenance:** the displayed full-suite spectra below are retained from the completed v0.6.0 run. The electromagnetic kernels are unchanged. The code is compatible with v0.6.3 and regenerates the results on Run All. Notebook 02 has a fresh v0.6.3 circular sweep; notebook 03 has a fresh independent-interface-mesh benchmark.

The displayed baseline outputs were computed with ARCS v0.6.0. The code now targets v0.6.3. Default impedance includes RW and indirect space charge; the free-space direct term is excluded. Complex outputs use the reversed convention requested by the user. Plots show signed Im(Z) directly. Finite-beta space-charge wakes use a two-sided DCT/DST inverse.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / "src/arcs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import arcs
import scipy
from arcs import (load_geometry, import_geometry, GeometryDocument, Metal, Wall,
                  impedance_to_wake, wake_to_impedance)
from arcs.reference import circular_exact, circular_maxwell_dipolar
from arcs.plotting import plot_longitudinal, plot_transverse
from arcs.geometry_input import _save_figure
from arcs.constants import C0, MU0
OUTPUT = ROOT / "results/notebook"
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 10, "figure.dpi": 110, "savefig.dpi": 160})
print("ARCS", arcs.__version__)
print("Module:", arcs.__file__)
print("NumPy", np.__version__, "SciPy", scipy.__version__)
if arcs.__version__ != "0.6.6" or Path(arcs.__file__).resolve().parent != (ROOT / "src/arcs").resolve():
    raise RuntimeError("Restart the notebook kernel to load this ARCS checkout.")
print("Output directory:", OUTPUT)
sys.path.insert(0, str(ROOT / "examples"))
from wideband import sweep, sweep_components, save_spectrum

from arcs import special as arcs_special
print("Green-function module:", arcs_special.__file__)
print("Stable K0(680+680j):", arcs_special.decaying_k(0, 680+680j))


## 1. Circular boundary mesh and the full frequency range

The main circle uses at least 256 boundary panels. This is a numerical demonstration; the separate release benchmark uses 512 panels on the identical 100-frequency grid. Refine panels, kernel integration, and material-profile parameters separately when assessing accuracy.

In [ ]:
MINIMUM_PANELS = 256
FREQUENCY_POINTS = 100
FREQUENCY_MIN = 10.0
FREQUENCY_MAX = 1e13
frequency = np.geomspace(FREQUENCY_MIN, FREQUENCY_MAX, FREQUENCY_POINTS)
assert len(frequency) == 100 and frequency[0] == 10. and frequency[-1] == 1e13
case = load_geometry(ROOT / "examples/geometries/bare_copper.json")
definition = case.to_dict()
definition["mesh"]["minimum_panels"] = MINIMUM_PANELS
case = load_geometry(definition)
solver = case.make_solver("maxwell", pipe_length_m=None)
mesh = solver.backend.panels
print("Frequency samples:", len(frequency), "Range [Hz]:", frequency[[0,-1]])
print("Boundary meshes:", len(case.material_panels()), "Actual panels:", len(mesh.panel_start))
figure = case.preview(OUTPUT / "circle_mesh.png", show_mesh=True)
display(figure); plt.close(figure)
response = sweep(solver, frequency, label="Circular copper",
                 output=OUTPUT / "circle_impedance.csv")
print("Calculated Z_parallel and complete 2x2 dipolar/quadrupolar tensors.")

In [ ]:
metal = Metal(5.8e7, "Cu")
z_reference = circular_exact(frequency, .03, Wall(metal))
figure = plot_longitudinal(frequency, response.longitudinal, reference=z_reference,
                          title="30 mm copper circle · full Maxwell longitudinal calculation")
_save_figure(figure, OUTPUT / "circle_longitudinal.png", dpi=160)
display(figure); plt.close(figure)
print("Largest complex relative discrepancy [%]:", 100*np.max(abs(response.longitudinal/z_reference - 1)))
print("Signed Im at the highest five sampled frequencies:", response.longitudinal[-5:].imag)
print("Largest absolute Im discrepancy [Ω/m]:", np.max(abs(response.longitudinal.imag-z_reference.imag)))
print("Largest Re-component discrepancy [%]:", 100*np.max(abs(response.longitudinal.real/z_reference.real - 1)))


The transverse expansion about the beam position is

$$Z_{\perp,i}/L = Z^{(0)}_{\perp,i}/L + \sum_j D_{ij}\,\Delta r_{s,j} + \sum_j Q_{ij}\,\Delta r_{w,j} + O(\Delta r^2).$$

Rows identify kick direction and columns identify displacement direction. **Dipolar** terms multiply the source offset; **quadrupolar** terms multiply the witness offset. Both coefficients are in Ω/m² per unit pipe length. A uniform length in metres multiplies them to give total dipolar/quadrupolar impedance in Ω/m.

For a centred circular pipe, horizontal and vertical dipoles agree, cross-plane terms vanish, and quadrupolar terms vanish. Raw numerical quadrupole residuals are retained. At large skin depth the dipole is not obtained by multiplying the monopole by the shallow-penetration factor.

In [ ]:
dip_reference = circular_maxwell_dipolar(frequency, .03, metal)
figure, axes = plt.subplots(1, 2, figsize=(10, 4.2), layout="constrained")
from arcs.plotting import _scale
for axis, component, label in zip(axes, (np.real, np.imag), ("Re", "Im")):
    reference_component = component(dip_reference)
    calculated_component = component(response.dipolar[:, 0, 0])
    axis.plot(frequency, reference_component, color="black", label="Independent full Maxwell m=1")
    axis.plot(frequency, calculated_component, "o", ms=3, label="ARCS dipolar xx")
    _scale(axis, np.r_[reference_component, calculated_component])
    axis.set_ylabel(f"{label}(Zdip/L) [Ω/m²]")
    axis.legend(fontsize=8)
figure.suptitle("Circular dipolar impedance · full Maxwell reference")
_save_figure(figure, OUTPUT / "circle_dipolar_reference.png", dpi=160)
display(figure); plt.close(figure)
figure = plot_transverse(frequency, response, title="Centred copper circle · Maxwell tensors")
_save_figure(figure, OUTPUT / "circle_transverse.png", dpi=160)
display(figure); plt.close(figure)
print("Dipolar complex relative discrepancy [%]:", 100*np.max(abs(response.dipolar[:,0,0]/dip_reference-1)))
print("Max |quadrupolar| / |dipolar|:", np.max(abs(response.quadrupolar)/abs(dip_reference[:,None,None])))

## 2. Import examples — geometry and numerical results

The three cases below each use the same 100-frequency grid. The cases are a rotated ellipse (JSON), a polygon coordinate table (CSV), and a sharp corner notch (DXF). All are assigned the same copper exterior, with source units stated for CSV. Each import is followed by its mesh, longitudinal plot and x/y dipolar/quadrupolar plots.

The DXF example retains its true straight-sided corners. Cross-plane tensors are saved as well; a rotated or asymmetric cross-section can have nonzero `xy` and `yx` terms.

## High-frequency response, including signed Im(Z)

The plots preserve the imaginary-part sign change. Dense cylindrical-mode lines provide a numerical reference curve; the BEM markers remain exactly the 100 requested frequencies. No reference fitting or rescaling is performed.

In [ ]:
sys.path.insert(0, str(ROOT / "examples"))
from benchmark import plot_restored_response
figure = plot_restored_response(frequency, response, z_reference, panels=len(mesh.panel_start))
_save_figure(figure, OUTPUT / "high_frequency_response.png", dpi=160)
display(figure); plt.close(figure)

In [ ]:
examples = [
    ("json_ellipse", ROOT / "examples/geometries/ellipse.json", None),
    ("csv_polygon", ROOT / "examples/geometries/custom_outline.csv", "mm"),
    ("dxf_notch", ROOT / "examples/geometries/cropped_notch.dxf", None),
]
imported_results = {}
for label, path, source_units in examples:
    imported = import_geometry(path, units=source_units, exterior="Cu")
    document = GeometryDocument(imported.geometry)
    document.set_region(len(document.geometry.regions)-1, "Cu", {"conductivity": 5.8e7})
    # These controls define the actual mesh for both preview and solve.
    document.set_mesh({"minimum_panels": 128, "max_panel_length": 5., "curve_tolerance": .05})
    geometry = document.geometry
    geometry.save(OUTPUT / f"{label}.json")
    print("\n", label, path.name)
    for note in imported.notes:
        print("Import note:", note)
    figure = geometry.preview(OUTPUT / f"{label}_mesh.png", show_mesh=True)
    display(figure); plt.close(figure)
    calculation = geometry.make_solver("maxwell", pipe_length_m=None)
    values = sweep(calculation, frequency, label=label, output=OUTPUT / f"{label}_impedance.csv")
    imported_results[label] = (geometry, values)
    figure = plot_longitudinal(frequency, values.longitudinal, title=f"{label} · bare copper Maxwell")
    _save_figure(figure, OUTPUT / f"{label}_longitudinal.png", dpi=160)
    display(figure); plt.close(figure)
    figure = plot_transverse(frequency, values, title=f"{label} · Maxwell transverse tensors")
    _save_figure(figure, OUTPUT / f"{label}_transverse.png", dpi=160)
    display(figure); plt.close(figure)
    columns, headers = [frequency, values.longitudinal.real, values.longitudinal.imag], ["f_Hz", "Re_long_Ohm_per_m", "Im_long_Ohm_per_m"]
    for tensor, name in ((values.dipolar, "dipolar"), (values.quadrupolar, "quadrupolar")):
        for i, first in enumerate("xy"):
            for j, second in enumerate("xy"):
                columns.extend([tensor[:,i,j].real, tensor[:,i,j].imag])
                headers.extend([f"Re_{name}_{first}{second}_Ohm_per_m2", f"Im_{name}_{first}{second}_Ohm_per_m2"])
    np.savetxt(OUTPUT / f"{label}_impedance.csv", np.column_stack(columns), delimiter=",", comments="", header=",".join(headers))
    print("Panels:", len(calculation.backend.panels.panel_start))
    print("Max |Qxx+Qyy| / max |D|:", np.max(abs(np.trace(values.quadrupolar,axis1=1,axis2=2)))/np.max(abs(values.dipolar)))

## 3. Independently shaped conducting coating interfaces

The reentrant aperture and coating outer outline below have different, varying thickness. Both are meshed and their Maxwell fields are matched. The source is inside the lower-left part of the aperture. This is an arbitrary-interface demonstration, not an analytical accuracy certificate. Smooth coatings are supported; air exterior is still rejected.

In [ ]:
layered = load_geometry(ROOT / "examples/geometries/nonuniform_coating.json")
figure = layered.preview(OUTPUT / "nonuniform_interfaces_mesh.png", show_mesh=True)
display(figure); plt.close(figure)
coating_frequency = frequency
coating_solver = layered.make_solver(pipe_length_m=None)
coating_response = sweep(coating_solver, coating_frequency, label="Nonuniform coating", output=OUTPUT / "nonuniform_coating_impedance.csv")
figure = plot_longitudinal(coating_frequency, coating_response.longitudinal,
                          title="Independent coating outline · full Maxwell BEM")
_save_figure(figure, OUTPUT / "nonuniform_coating_longitudinal.png")
display(figure); plt.close(figure)
figure = plot_transverse(coating_frequency, coating_response,
                        title="Independent coating outline · dipolar/quadrupolar")
_save_figure(figure, OUTPUT / "nonuniform_coating_transverse.png")
display(figure); plt.close(figure)
print("Physical boundary meshes:", len(layered.boundaries))

## 4. Finite-beta full Maxwell example

The beta = 0.7 circle also uses the 100-frequency grid. Its resistive-wall reaction can become exponentially small at high frequency and underflow to zero; the error calculation explicitly counts these points rather than dividing by a zero reference. The same-beta PEC pipe defines the RW reference.

In [ ]:
from arcs import MaxwellSolver, Roughness
from arcs.reference import circular_maxwell_quadrupolar
slow_data = case.to_dict()
slow_data["beta"] = 0.7
slow_data["mesh"]["minimum_panels"] = 256
slow = load_geometry(slow_data).make_solver(pipe_length_m=None)
slow_f = frequency
slow_parts = sweep_components(slow, slow_f, label="Finite beta = 0.7: all contributions", output=OUTPUT / "finite_beta_components.csv")
slow_response = slow_parts["wall"]
save_spectrum(OUTPUT / "finite_beta_impedance.csv", slow_f, slow_response)
slow_reference = circular_exact(slow_f, .03, Wall(metal), beta=0.7)
figure = plot_longitudinal(slow_f, slow_response.longitudinal, reference=slow_reference,
                          title="Finite-beta circular Maxwell fields · beta = 0.7")
_save_figure(figure, OUTPUT / "finite_beta_longitudinal.png")
display(figure); plt.close(figure)
figure = plot_transverse(slow_f, slow_response, title="Finite-beta circle · nonzero quadrupolar terms")
_save_figure(figure, OUTPUT / "finite_beta_transverse.png")
display(figure); plt.close(figure)
quad_reference = circular_maxwell_quadrupolar(slow_f, .03, Wall(metal), beta=.7)
resolved = abs(slow_reference) > 1e-250
print("Finite-beta numerical comparison points:", np.count_nonzero(resolved), "/", len(slow_f))
print("Reference below 1e-250 Ω/m:", np.count_nonzero(~resolved))
print("Finite-beta maximum error on resolved points [%]:", 100*np.max(abs(slow_response.longitudinal[resolved]/slow_reference[resolved]-1)))
index = int(np.argmin(abs(np.log(slow_f/1e9))))
print("Qxx and reference near 1 GHz at", slow_f[index], "Hz:", slow_response.quadrupolar[index,0,0], quad_reference[index])

## Resolve the three wall contributions

The PEC image field is evaluated with the same aperture boundary mesh. Circular formulas are independent checks. The 10 Hz–10 THz grids are identical for all contributions. Tiny quadrupoles and exponentially screened values are audited with absolute errors or explicitly resolved frequency ranges.

In [ ]:
from space_charge_benchmark import audit_components, plot_components
space_charge_summary = audit_components(slow_f, slow_parts, .03, metal, .7, OUTPUT / "space_charge_summary.json")
figure = plot_components(slow_f, slow_parts, .03, metal, .7)
_save_figure(figure, OUTPUT / "ARCS_v0.6.0_space_charge_components.png", dpi=180)
display(figure); plt.close(figure)
print(json.dumps(space_charge_summary, indent=2))
print("Default output = RW + indirect SC. Direct SC is excluded.")


## 5. Roughness across the same 100 frequencies — prioritize Im(Z)

The **truncated** Gaussian conductivity-gradient BEM uses Rq = 0.3 μm, 32 slices and an air-side extent of 5 RMS heights, with the mean radius held fixed. It is compared with identical discrete cylindrical Maxwell layers and continuous cylindrical Maxwell integration at the same cutoff. A separate continuous 10-RMS profile shows the unresolved cutoff error in Im(Z) and the reactive increment.

The wider BEM sweep found that extending the near-insulating tail to 10 RMS heights destabilizes the current two-field layer elimination. ARCS rejects BEM extents above 5 rather than returning misleading impedances. Mesh construction for those extents remains available. The archived failure endpoint check is included in the release; this is an outstanding numerical limitation, not a correction of the complete Gaussian model. The supplied reduced ODE is not a universal full-band reference.

For this centred β=1 circular case, exact quadrupolar coefficients are zero. The small raw quadrupolar values plotted below are numerical residuals; compare their scale with the dipolar coefficients. This 64-panel demonstration has a maximum same-layer discrepancy of about 1.00% longitudinal and 3.46% dipolar across the entire grid. Refine boundary panels separately from the conductivity slices.


In [ ]:
from arcs import Layer
from arcs.roughness import gradient_layers
from arcs.reference import circular_layered_dipolar
sys.path.insert(0, str(ROOT / "examples/roughness_audit"))
import audit_imaginary as gaussian_check
ROUGH_SLICES = 32
ROUGH_EXTENT = 5.0
rough = Roughness(0.3e-6, slices=ROUGH_SLICES, extent_sigma=ROUGH_EXTENT)
mean_radius = gaussian_check.MEAN_RADIUS
aperture_radius = mean_radius - rough.extent_sigma*rough.rms_height
rough_definition = case.to_dict()
rough_definition["aperture"]["radius"] = aperture_radius / case.unit_scale
rough_definition["mesh"].update(minimum_panels=64, max_panel_length=5., curve_tolerance=.05)
rough_definition["roughness"] = {"model":"conductivity_gradient", "rms_height":rough.rms_height/case.unit_scale,
                                "slices":rough.slices, "extent_sigma":rough.extent_sigma}
rough_case = load_geometry(rough_definition)
rough_case.save(OUTPUT / "roughness_case.json")
actual_interfaces = rough_case.material_panels()
print("Roughness boundary meshes:", len(actual_interfaces))
print("Total panels:", sum(len(p.panel_start) for _, p in actual_interfaces))
print("Mean wall radius [m]:", mean_radius, "Aperture radius [m]:", aperture_radius)
rough_f = frequency
rough_solver = rough_case.make_solver(pipe_length_m=None)
rough_response = sweep(rough_solver, rough_f, label="Gaussian roughness",
                       output=OUTPUT / "roughness_impedance.csv")
depth, media = gradient_layers(Wall(metal), rough)
radial_wall = Wall(metal, tuple(Layer(material, thickness)
                  for material, thickness in zip(media[:-1], np.diff(depth))))
rough_reference = circular_exact(rough_f, aperture_radius, radial_wall)
rough_dip_reference = circular_layered_dipolar(rough_f, aperture_radius, radial_wall)
continuous_reference = np.array([gaussian_check.continuous_gradient(f, np.array([ROUGH_EXTENT]))[0]
                                 for f in rough_f])
continuous_deep = np.array([gaussian_check.continuous_gradient(f, np.array([10.]))[0]
                            for f in rough_f])
smooth_mean = circular_exact(rough_f, mean_radius, Wall(metal))
figure = plot_longitudinal(rough_f, rough_response.longitudinal, reference=rough_reference,
                          reference_label="Same discrete circular Maxwell layers",
                          title="Gaussian conductivity gradient · 100 BEM frequencies")
_save_figure(figure, OUTPUT / "roughness_longitudinal.png")
display(figure); plt.close(figure)
figure = plot_transverse(rough_f, rough_response, title="Gaussian roughness · full-band BEM tensors")
_save_figure(figure, OUTPUT / "roughness_transverse.png")
display(figure); plt.close(figure)
figure, axes = plt.subplots(1, 2, figsize=(11, 4.3), layout="constrained")
for values, label, style in ((rough_response.longitudinal, "ARCS BEM, 32 slices; cutoff 5 Rq", "o"),
                             (rough_reference, "Circular matching, same 32 slices", "-"),
                             (continuous_reference, "Continuous Maxwell, cutoff 5 Rq", "--"),
                             (continuous_deep, "Continuous Maxwell, cutoff 10 Rq", ":")):
    axes[0].plot(rough_f, values.imag, style, ms=3, label=label)
    axes[1].plot(rough_f, values.imag-smooth_mean.imag, style, ms=3, label=label)
for axis in axes:
    _scale(axis, np.r_[rough_response.longitudinal.imag, continuous_reference.imag])
    axis.axhline(0, color="grey", lw=.6); axis.legend(fontsize=7)
axes[0].set(title="Signed imaginary impedance", ylabel="Im(Z∥/L) [Ω/m]")
axes[1].set(title="Roughness-induced reactive increment", ylabel="ΔIm(Z∥/L) [Ω/m]")
_save_figure(figure, OUTPUT / "roughness_imaginary_full_band.png")
display(figure); plt.close(figure)
np.savetxt(OUTPUT / "roughness_references.csv", np.c_[rough_f, rough_reference.real, rough_reference.imag,
    continuous_reference.real, continuous_reference.imag, continuous_deep.real, continuous_deep.imag, rough_dip_reference.real, rough_dip_reference.imag,
    smooth_mean.real, smooth_mean.imag], delimiter=",", comments="", header=
    "f_Hz,Re_discrete_Maxwell,Im_discrete_Maxwell,Re_continuous_Maxwell,Im_continuous_Maxwell,Re_continuous_cut10,Im_continuous_cut10,Re_dipolar_Maxwell,Im_dipolar_Maxwell,Re_smooth_mean,Im_smooth_mean")
print("Same-layer longitudinal maximum complex error [%]:", 100*np.max(abs(rough_response.longitudinal/rough_reference-1)))
print("Same-layer dipolar maximum complex error [%]:", 100*np.max(abs(rough_response.dipolar[:,0,0]/rough_dip_reference-1)))
print("Largest absolute Im error to same layers [Ω/m]:", np.max(abs(rough_response.longitudinal.imag-rough_reference.imag)))
print("Largest absolute Im difference to continuous profile [Ω/m]:", np.max(abs(rough_response.longitudinal.imag-continuous_reference.imag)))
print("Largest absolute Im cutoff difference, continuous 5 vs 10 Rq [Ω/m]:", np.max(abs(continuous_reference.imag-continuous_deep.imag)))


In [ ]:
from IPython.display import Image
legacy_audit = ROOT / "examples/roughness_audit/results/ARCS_Imaginary_Roughness_Audit.png"
print("Historical cutoff audit: saved v0.4.0 BEM values at −5Rq; independent continuous Maxwell cutoff study.")
display(Image(filename=str(legacy_audit)))
print("Source and raw audit data:", legacy_audit.parents[1])

## 6. Your DCT-I/DST-I conversion — independent resonator check

The matched kernels follow `wakeSolver_v3.py`:

- Longitudinal: cosine → Re(Z), sine → Im(Z).
- Transverse: sine → Re(Z), cosine → Im(Z).
- Stored longitudinal origin: W(0)=W(0+)/2.
- Native spacing: dt=1/(2 fmax), with N frequency points giving N time points.

The native DCT convention uses exp(−iωt) in the wake integral. ARCS's impedance convention uses exp(+iωt), with its corresponding transverse −i prefactor. Use **`convention="arcs"`** for ARCS arrays; the adapter explicitly conjugates the spectrum. It changes no amplitude or time origin.

First check both conversion planes against an analytical damped resonator, independently of the Maxwell geometry calculation. The transverse converter also applies to a quadrupolar coefficient array; it does not itself calculate transverse fields.

In [ ]:
f_transform = np.linspace(0., 1e12, 32769)
omega = 2*np.pi*f_transform
wr = 2*np.pi*5e9
quality = 2.
alpha = wr/(2*quality)
wd = np.sqrt(wr**2-alpha**2)
denominator = wr**2-omega**2+2j*alpha*omega
z_long_resonator = 1j*omega*(100*wr/quality)/denominator
z_trans_resonator = 1j*1e14*wd/denominator
figure, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
resonator_errors = {}
for axis, plane, spectrum in zip(axes, ("longitudinal", "transverse"), (z_long_resonator, z_trans_resonator)):
    time, wake = impedance_to_wake(f_transform, spectrum, plane=plane, convention="dct")
    if plane == "longitudinal":
        expected = 100*wr/quality*np.exp(-alpha*time)*(np.cos(wd*time)-alpha/wd*np.sin(wd*time))
        expected[0] *= .5
    else:
        expected = 1e14*np.exp(-alpha*time)*np.sin(wd*time)
    view = time <= 1e-9
    scale = np.max(abs(expected[view]))
    axis.plot(time[view]*1e9, expected[view]/scale, color="black", label="Analytical resonator")
    axis.plot(time[view]*1e9, wake[view]/scale, "--", label="DCT-I / DST-I")
    axis.set(title=plane, xlabel="Time behind source [ns]", ylabel="Wake / analytical peak")
    axis.grid(alpha=.2); axis.legend(fontsize=8)
    error = float(np.linalg.norm(wake[view]-expected[view])/np.linalg.norm(expected[view]))
    resonator_errors[plane] = error
    print(plane, "relative L2 error:", error)
_save_figure(figure, OUTPUT / "dct_dst_resonators.png", dpi=160)
display(figure); plt.close(figure)

A discrete round trip checks normalization and Fourier signs, including both impedance components. The cosine/DST inverse uses Re(Z) and causality as in your implementation. An arbitrary inconsistent Re/Im input is not repaired by this inverse.

In [ ]:
t_check = np.linspace(0., 2e-9, 513)
w_check = np.exp(-t_check/2e-10)*np.cos(2*np.pi*5e9*t_check)
w_check[0] *= .5
f_check, z_check = wake_to_impedance(t_check, w_check, convention="dct")
t_roundtrip, w_roundtrip = impedance_to_wake(f_check, z_check, convention="dct")
print("Matched longitudinal round-trip relative L2:", np.linalg.norm(w_roundtrip-w_check)/np.linalg.norm(w_check))
figure, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
axes[0].plot(f_check/1e9, z_check.real, label="Re"); axes[1].plot(f_check/1e9, z_check.imag, label="signed Im")
for axis in axes:
    axis.set(xlabel="Frequency [GHz]", ylabel="Impedance [Ω/m]"); axis.grid(alpha=.2); axis.legend()
figure.suptitle("DCT/DST forward conversion of the causal wake")
_save_figure(figure, OUTPUT / "dct_dst_impedance_components.png", dpi=160)
display(figure); plt.close(figure)

## 7. ARCS → your DCT/DST — finite-band Maxwell wakes

Compute an independent uniform frequency grid up to 10 THz up to 10 THz and apply `convention="arcs"`. The DC entries use the exact centred bare-circle limits; the field solver itself does not evaluate DC. This cell requests at least 128 panels; the maximum panel length can increase the actual count. The full-band impedance audit above uses at least 256 panels.

The analytical circular spectra undergo the same transform as a separate check. **These are finite-band reconstructions**, not a converged infinite-band point wake. The native timestep is 0.1 ps; the frequency spacing limits the available time window. Increase bandwidth, frequency density and boundary resolution separately before using a short-bunch wake. A generic geometry needs its own justified DC values.

In [ ]:
uniform_f = np.linspace(0., 1e13, 257)
wake_definition = case.to_dict()
wake_definition["mesh"]["minimum_panels"] = 128
wake_solver = load_geometry(wake_definition).make_solver("maxwell", pipe_length_m=None)
uniform_response = sweep(wake_solver, uniform_f[1:], label="Uniform grid for DCT/DST")
z_uniform = np.r_[0j, uniform_response.longitudinal]
# Full beta=1 centred circular m=1 limit: Zdip(0) = +i*Z0/(2*pi*b²).
dipolar_dc = 1j*C0*MU0/(2*np.pi*.03**2)
z_dip_uniform = np.r_[dipolar_dc, uniform_response.dipolar[:,0,0]]
z_exact_uniform = circular_exact(uniform_f, .03, Wall(metal))
z_dip_exact = np.r_[dipolar_dc, circular_maxwell_dipolar(uniform_f[1:], .03, metal)]
figure, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
converted = []
for axis, plane, spectrum, reference_spectrum in zip(axes, ("longitudinal", "transverse"),
        (z_uniform, z_dip_uniform), (z_exact_uniform, z_dip_exact)):
    time, wake = impedance_to_wake(uniform_f, spectrum, plane=plane, convention="arcs")
    _, reference_wake = impedance_to_wake(uniform_f, reference_spectrum, plane=plane, convention="arcs")
    view = time <= 10e-12
    axis.plot(time[view]*1e12, reference_wake[view], color="black", label="Circular Maxwell spectrum → same transform")
    axis.plot(time[view]*1e12, wake[view], "--", label="ARCS → your DCT/DST")
    axis.set(title=plane, xlabel="Time behind source [ps]", ylabel="Finite-band wake [V/C/m]" if plane=="longitudinal" else "Finite-band coefficient [V/C/m²]")
    axis.grid(alpha=.2); axis.legend(fontsize=7)
    converted.append(wake)
    print(plane, "finite-band wake discrepancy L2:", np.linalg.norm(wake-reference_wake)/np.linalg.norm(reference_wake))
figure.suptitle("Circular full Maxwell → DCT/DST · finite bandwidth 0–10 THz")
_save_figure(figure, OUTPUT / "maxwell_band_limited_wakes.png", dpi=160)
display(figure); plt.close(figure)
np.savetxt(OUTPUT / "maxwell_band_limited_wakes.csv", np.column_stack((time,*converted)), delimiter=",", comments="", header="time_s,long_wake_V_per_C_per_m,dipolar_wake_V_per_C_per_m2")
print("Native dt [s]:", time[1]-time[0])

## Reuse the workflow

- GUI: `python launch_gui.py`; apply β/roughness and mesh settings, then save the JSON case.
- Saved case: `python examples/run_case.py my_pipe.json --beta 0.7 --f-max 1e9`.
- Full β=1 audit: `python examples/benchmark.py --panels 512`.
- Finite-beta audit: `python examples/finite_beta_benchmark.py`.
- Roughness audit: `python examples/roughness_benchmark.py --low-frequency`.
- `make_solver("maxwell")` is the field backend. MQS is removed.
- Smooth conducting coatings are supported. Air exterior and z-varying geometric roughness remain unsupported.
- See `docs/maxwell.md`, `docs/roughness.md`, `docs/transverse.md`, and `docs/wakes.md`.
- Each geometry sweep above exports all 100 directly calculated frequencies and both components.
- The Gaussian cutoff audit is in `examples/roughness_audit/`; keep its historical reference settings explicit.
- DCT/DST requires the separate uniform frequency grid; the 100-point logarithmic impedance grid is not passed directly to the transform.


## Finite-beta indirect-space-charge wakes on signed time

A Re-only causal inverse would erase this purely imaginary contribution. Use `causal=False` to retain both frequency components through DCT-I/DST-I. No causal projection is applied. These are finite-band waveforms, compared with an independent circular spectrum on the same grid.

In [ ]:
from arcs.reference import circular_indirect_space_charge, circular_indirect_dipolar
image_f = np.linspace(0., 5e10, 1025)
image_solver = MaxwellSolver(case.to_panels(), beta=.7)
image_response = image_solver.transverse(image_f[1:], component="indirect_space_charge")
image_spectra = (np.r_[0j, image_response.longitudinal],
                 np.r_[circular_indirect_dipolar(0., .03, .7), image_response.dipolar[:,0,0]])
image_references = (circular_indirect_space_charge(image_f,.03,.7), circular_indirect_dipolar(image_f,.03,.7))
figure, axes = plt.subplots(1, 2, figsize=(11,4.2), layout="constrained")
image_wake_errors = {}
for axis, plane, spectrum, reference in zip(axes, ("longitudinal","transverse"), image_spectra, image_references):
    image_time, image_wake = impedance_to_wake(image_f,spectrum,plane=plane,causal=False)
    _, image_exact_wake = impedance_to_wake(image_f,reference,plane=plane,causal=False)
    view = abs(image_time) < 1e-9
    axis.plot(image_time[view]*1e9,image_exact_wake[view],color="black",label="Circular image formula → DCT/DST")
    axis.plot(image_time[view]*1e9,image_wake[view],"--",label="BEM image field → DCT/DST")
    axis.set(xlabel="Signed time behind source [ns]",ylabel="Wake [V/C/m]" if plane=="longitudinal" else "Dipolar wake [V/C/m²]",title=plane)
    axis.grid(alpha=.2);axis.legend(fontsize=8)
    image_wake_errors[plane] = float(np.linalg.norm(image_wake-image_exact_wake)/np.linalg.norm(image_exact_wake))
    np.savetxt(OUTPUT/f"indirect_SC_{plane}_signed_wake.csv",np.c_[image_time,image_wake,image_exact_wake],delimiter=",",comments="",header="time_s,BEM_wake,circular_reference_wake")
_save_figure(figure,OUTPUT/"ARCS_v0.6.0_signed_space_charge_wakes.png",dpi=180)
display(figure);plt.close(figure)
(OUTPUT/"space_charge_wake_summary.json").write_text(json.dumps({"beta":.7,"frequency_band_Hz":[0.,5e10],"uniform_samples":len(image_f),"relative_L2_errors":image_wake_errors,"two_sided":True},indent=2)+"\n")
print("Signed-space-charge wake errors:", image_wake_errors)
